# 📥 Phase 0: MS Lesion Dataset Downloader (aria2c Optimized)
This notebook downloads all required datasets directly to your Google Drive using **aria2c** for maximum speed and reliability. Run this **once** before starting training.

In [ ]:
from google.colab import drive
import os

# 1. Setup Drive
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/brain_dataset"

# 2. Install aria2 for high-speed, reliable downloads
!apt-get install -y aria2

# 3. Define Dataset Structure
datasets = {
    "MSLesSeg": {
        "path": os.path.join(BASE_PATH, "MSLesSeg"),
        "links": [
            ("https://springernature.figshare.com/ndownloader/files/52771814", "MSLesSeg_Part1.zip"),
            ("https://springernature.figshare.com/ndownloader/files/53623946", "MSLesSeg_Part2.zip"),
            ("https://springernature.figshare.com/ndownloader/files/53623967", "MSLesSeg_Part3.zip"),
            ("https://springernature.figshare.com/ndownloader/files/54605591", "MSLesSeg_Part4.zip"),
            ("https://springernature.figshare.com/ndownloader/files/54605609", "MSLesSeg_Part5.zip")
        ]
    },
    "Mendeley": {
        "path": os.path.join(BASE_PATH, "Mendeley"),
        "links": [
            ("https://data.mendeley.com/public-files/datasets/8bctsm8jz7/files/9356efeb-dcd8-4213-a2d4-8febe9f1a5db/file_downloaded", "Mendeley_Data.zip")
        ]
    },
    "Long-MR-MS": {
        "path": os.path.join(BASE_PATH, "Long-MR-MS"),
        "links": [
            ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_demographics.csv", "demographics.csv"),
            ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient01-05.zip", "Long_Part1.zip"),
            ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient06-10.zip", "Long_Part2.zip"),
            ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient11-15.zip", "Long_Part3.zip"),
            ("https://lit.fe.uni-lj.si/data/research/resources/long-MR-MS/long-MR-MS_patient16-20.zip", "Long_Part4.zip")
        ]
    }
}

# 4. Execute Downloads
import requests
from tqdm.notebook import tqdm

def download_robustly(url, target_dir, filename):
    target_file = os.path.join(target_dir, filename)
    if os.path.exists(target_file) and os.path.getsize(target_file) > 0:
        print(f"  ✅ {filename} already exists.")
        return
        
    print(f"  🚀 Robustly Downloading: {filename}...")
    session = requests.Session()
    try:
        response = session.get(url, allow_redirects=True, stream=True)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        
        with open(target_file, "wb") as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=filename, leave=False) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        print(f"  ✨ Saved {filename}")
    except Exception as e:
        print(f"  ❌ Error downloading {filename}: {e}")

for name, info in datasets.items():
    print(f"\n📂 Organizing {name}...")
    os.makedirs(info['path'], exist_ok=True)
    
    for link, filename in info['links']:
        if name == "MSLesSeg":
            # Use robust Python downloader for Figshare links
            download_robustly(link, info['path'], filename)
        else:
            # Use aria2c for other datasets
            target_file = os.path.join(info['path'], filename)
            if not os.path.exists(target_file) or os.path.getsize(target_file) == 0:
                print(f"  🚀 aria2c Downloading: {filename}...")
                !aria2c -x 16 -s 16 -k 1M "{link}" -d "{info['path']}" -o "{filename}"
            else:
                print(f"  ✅ {filename} already exists.")

print("\n🚀 FULL PIPELINE OPTIMIZED: All datasets are now successfully on your Drive.")